# Setting up

In [1]:
import requests
import json
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
OMDB_API_KEY = os.getenv("OMDB_API_KEY")

if OMDB_API_KEY:
    print("✅ OMDB API key loaded successfully!")
else:
    print("❌ OMDB API key not found.")

✅ OMDB API key loaded successfully!


## Step 1: Loading TMDB Data

The OMDb data collection step relies on the movie information collected from TMDB in the previous notebook.

I first load the raw TMDB movie JSON files generated in NB01a. These files contain the movie records required for OMDb requests, including the `imdb_id` used as the identifier for fetching additional information.

Before starting the API requests, I inspect the loaded data structure to confirm that the JSON files were successfully imported and contain the expected movie records.

In [2]:
with open("../data/raw/tmdb_indie_movies.json", 'r') as f:
    indie_movies = json.load(f) 

In [3]:
type(indie_movies)
indie_movies[0]

{'id': 293660,
 'title': 'Deadpool',
 'imdb_id': 'tt1431045',
 'release_date': '2016-02-09',
 'production_companies': [{'id': 25,
   'logo_path': '/nM2MfoMqzJQRiSynsDabOtFKetD.png',
   'name': '20th Century Fox',
   'origin_country': 'US'},
  {'id': 431,
   'logo_path': '/6dcR1MbRqYgt3jUVYxkHe68GFnZ.png',
   'name': "The Donners' Company",
   'origin_country': 'US'},
  {'id': 28788,
   'logo_path': '/Aqomtf9oh5dKtxBNEagkdlp3aGv.png',
   'name': 'Genre Films',
   'origin_country': 'US'},
  {'id': 7505,
   'logo_path': '/837VMM4wOkODc1idNxGT0KQJlej.png',
   'name': 'Marvel Entertainment',
   'origin_country': 'US'}],
 'budget': 58000000,
 'revenue': 782837347,
 'vote_average': 7.624,
 'vote_count': 32966}

## Step 2: Fetching OMDb Data

The OMDb API is used to retrieve information that is not available from TMDB, especially professional critic ratings (`Metascore`).

Each OMDb request requires an IMDb ID, which is obtained from the TMDB dataset.

In [4]:
def fetch_omdb_data(imdb_id):
    """
    Fetch rating data for a single movie from OMDb using its IMDb ID.
    """ 
    URL = "http://www.omdbapi.com"
    params = {"apikey":"OMDB_API_KEY",
             "i": imdb_id
             }
    response = requests.get(URL, params=params)
    if response.status_code != 200:
        print("request wasnt successful!")
        return None
    return response.json()

### Test one API call
Before collecting the full dataset, I test the API request with a single movie ID (because there's a daily 1000 free API calls limit) to verify that:
- the API connection works correctly
- the returned response contains the expected fields

In [5]:
test = fetch_omdb_data("tt7286456")
test

{'Title': 'Joker',
 'Year': '2019',
 'Rated': 'R',
 'Released': '04 Oct 2019',
 'Runtime': '122 min',
 'Genre': 'Crime, Drama, Thriller',
 'Director': 'Todd Phillips',
 'Writer': 'Todd Phillips, Scott Silver, Bob Kane',
 'Actors': 'Joaquin Phoenix, Robert De Niro, Zazie Beetz',
 'Plot': 'Arthur Fleck, a party clown and a failed stand-up comedian, leads an impoverished life with his ailing mother. However, when society shuns him and brands him as a freak, he decides to embrace the life of chaos in Gotham City.',
 'Language': 'English, German',
 'Country': 'United States, Canada, Australia',
 'Awards': 'Won 2 Oscars. 120 wins & 247 nominations total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BNzY3OWQ5NDktNWQ2OC00ZjdlLThkMmItMDhhNDk3NTFiZGU4XkEyXkFqcGc@._V1_QL75_UX380_CR0,0,380,562_.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.3/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '68%'},
  {'Source': 'Metacritic', 'Value': '59/100'}],
 'Metascore': '59',
 'imd

### Select necessary columns (like in NB01a)
The OMDb response contains more information than required for this project, so only the relevant fields are retained:

- `imdbID`
- `Title`
- `Year`
- `Genre`
- `Metascore`
- `imdbRating`

The collection process is then applied to all movies in the dataset.

In [10]:
omdb_indie_data =[]
keep_omdb_fields = [
    "imdbID",
    "Title",
    "Year",
    "Genre",
    "Metascore",
    "imdbRating"
]


### Fetching indie_movies data from omdb

In [11]:
for movie in indie_movies:
    movie_imdb_id = movie["imdb_id"]
    omdb_result = fetch_omdb_data(movie_imdb_id)
    selected_movie_detail = {
        field: omdb_result[field] for field in keep_omdb_fields
    }
    omdb_indie_data.append(selected_movie_detail)

### Inspect the returned result

In [23]:
len(omdb_indie_data)

668

In [24]:
omdb_indie_data[0]

{'imdbID': 'tt1431045',
 'Title': 'Deadpool',
 'Year': '2016',
 'Genre': 'Action, Comedy, Sci-Fi',
 'Metascore': '65',
 'imdbRating': '8.0'}

## Step 3: Save the raw data to json file

In [25]:
with open("../data/raw/omdb_indie_movies.json","w") as f:
        json.dump(omdb_indie_data,f,indent=2)

## Step 4: Same procedure for major company

In [28]:
with open("../data/raw/tmdb_major_movies.json", 'r') as f:
    major_movies = json.load(f) 

omdb_major_data =[]
for movie in major_movies:
    movie_imdb_id = movie["imdb_id"]
    omdb_result = fetch_omdb_data(movie_imdb_id)
    selected_movie_detail = {
        field: omdb_result[field] for field in keep_omdb_fields
    }
    omdb_major_data.append(selected_movie_detail)

In [29]:
len(omdb_major_data)

209

In [30]:
omdb_major_data[0]

{'imdbID': 'tt7286456',
 'Title': 'Joker',
 'Year': '2019',
 'Genre': 'Crime, Drama, Thriller',
 'Metascore': '59',
 'imdbRating': '8.3'}

In [31]:
with open("../data/raw/omdb_major_movies.json","w") as f:
        json.dump(omdb_major_data,f,indent=2)

### FYI：Note on API Management

Although the major and non-major datasets could have been processed together in a **single loop** to avoid code replication, I collected them separately intentionally since OMDb has a daily API request limit, separating the two batches makes the data collection process easier to monitor and reduces the risk of losing the entire collection if an issue occurs. 

As there are only two groups, this approach adds little extra complexity while making the workflow more manageable.